# 2.04 Voyage AI Embeddings

Voyage AI의 `voyage-3`은 **RAG 품질**에 특화된 범용 임베더다. MTEB·KILT·BEIR 기준으로 OpenAI `text-embedding-3-large`와 경합 또는 앞서며, 한국어를 포함한 다국어 품질도 양호하다.
Cohere와 함께 **Rerank API**를 함께 제공해 2단계 파이프라인을 쉽게 구성할 수 있다.

## 학습 목표

- `VoyageAIEmbeddings`로 쿼리·문서를 벡터화한다
- `input_type`을 전환해 RAG 비대칭 인덱싱을 한다
- `voyage-3` vs `voyage-3-lite` vs `voyage-code-3`의 선택 기준을 안다
- Voyage Rerank와 결합한 2단계 검색의 구성을 본다

## 언제 쓰나

- **검색 품질을 최대화**하고 싶은 RAG 파이프라인
- **코드 검색**에 특화된 `voyage-code-3`이 필요한 개발자 도구
- **Rerank API**를 같은 공급자로 묶어 레이턴시를 줄이고 싶을 때

## 2.04.1 환경 설정

필요 패키지: `langchain-voyageai`. `.env`에 `VOYAGE_API_KEY`가 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-voyageai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("VOYAGE_API_KEY"), "VOYAGE_API_KEY 필요"

## 2.04.2 기본 사용법

`voyage-3`을 기본으로 사용한다.

In [ ]:
from langchain_voyageai import VoyageAIEmbeddings

embeddings = VoyageAIEmbeddings(model="voyage-3")

q_vec = embeddings.embed_query("Voyage AI 임베딩은 무엇인가?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "voyage-3은 1024차원 임베딩을 만든다.",
    "BEIR, MTEB에서 상위권 성능을 보인다.",
    "코드 전용 변형 voyage-code-3이 있다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.04.3 차원 · 비용 · 다국어 성능

| 모델 | 차원 | 최대 토큰 | 용도 |
|------|------|----------|------|
| `voyage-3` | 1024 | 32000 | 일반 RAG (권장 기본) |
| `voyage-3-lite` | 512 | 32000 | 저지연/저비용 |
| `voyage-code-3` | 1024 | 32000 | 코드 의미 검색 |
| `voyage-multilingual-2` | 1024 | 32000 | 다국어 강화 |
| `voyage-law-2` | 1024 | 16000 | 법률 도메인 |

- 한국어: `voyage-multilingual-2`가 더 정확하지만 일반 용도엔 `voyage-3`로도 충분
- 32k 토큰 입력은 본 공급자의 강점 — 긴 문서를 단일 임베딩으로 처리 가능
- 가격은 토큰 단위. 공식 대시보드 참조

## 2.04.4 `input_type` 비대칭 인덱싱

Voyage는 쿼리/문서 임베딩을 **서로 다른 프로젝션**으로 생성한다. `input_type="query"` 또는 `"document"`로 지정.

In [ ]:
doc_emb = VoyageAIEmbeddings(model="voyage-3", input_type="document")
query_emb = VoyageAIEmbeddings(model="voyage-3", input_type="query")

corpus = [
    "LangChain의 RAG 파이프라인은 임베딩→검색→생성 순이다.",
    "Voyage AI는 검색 품질에 특화된 임베딩을 제공한다.",
]
doc_vecs = doc_emb.embed_documents(corpus)
q = query_emb.embed_query("검색 품질을 올리는 임베더")
print("doc0 dim:", len(doc_vecs[0]), "| query dim:", len(q))

## 2.04.5 벡터스토어 연동 미리보기

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=corpus,
    embedding=doc_emb,
    collection_name="demo_voyage",
)
hits = store.similarity_search("검색 품질", k=2)
for h in hits:
    print("-", h.page_content)

## 2.04.6 Voyage Rerank — 2단계 검색

Voyage는 `rerank-2` / `rerank-2-lite` 모델을 제공한다. 임베딩 상위 k 후 교차 주의로 재정렬해 최종 품질을 끌어올린다.
LangChain은 `VoyageAIRerank` 래퍼를 제공한다 (자세한 활용은 `../05_retrievers/`).

In [ ]:
from langchain_voyageai import VoyageAIRerank
from langchain_core.documents import Document

reranker = VoyageAIRerank(model="rerank-2", top_k=2)

candidates = [Document(page_content=t) for t in [
    "파이썬은 인터프리터 언어다.",
    "Voyage Rerank는 상위 후보를 재정렬한다.",
    "임베딩 유사도는 1차 필터로 쓰인다.",
]]
ranked = reranker.compress_documents(candidates, query="검색 결과 재정렬")
for d in ranked:
    print("-", d.page_content)

## 체크리스트

- [ ] 일반 RAG은 `voyage-3`, 코드 검색은 `voyage-code-3`, 다국어 중시는 `voyage-multilingual-2`
- [ ] 쿼리/문서에 각각 `input_type` 지정
- [ ] 32k 토큰 입력 한도를 고려해 청크 크기 재설계 가능한지 확인
- [ ] Rerank와 묶어 2단계 파이프라인으로 최종 품질 확보

## 다음

- `05_ollama.ipynb` — 로컬 임베더 `nomic-embed-text`
- `../05_retrievers/` — Voyage Rerank로 `ContextualCompressionRetriever` 구성

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/voyageai
- Voyage 모델: https://docs.voyageai.com/docs/embeddings
- Rerank: https://docs.voyageai.com/docs/reranker